> **⚙️ Setup cell added by [doQumentation](https://doqumentation.org)**
>
> The code cell below was added automatically to install required packages
> (skipped if already installed, e.g. on Binder/Code Engine).
> It also contains a commented-out template for IBM Quantum credentials.
> [Learn more about automatic modifications.](https://doqumentation.org/about/code-modifications)

In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Introduction to primitives

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>

<span id="qpu-access-patterns"></span>
## Why did Qiskit introduce primitives?

Similar to the early days of classical computers, when developers had to manipulate CPU registers directly, the early interface to QPUs simply returned the raw data from the control electronics.
This was not a big issue when QPUs lived in labs and only allowed direct access by researchers.
Acknowledging that most developers would not and should not be familiar with distilling such raw data into 0s and 1s, Qiskit introduced `backend.run`, a first abstraction for accessing QPUs in the cloud. This allowed developers
to operate on a familiar data format and focus on the bigger picture.

As access to QPUs became more widespread, and with more quantum algorithms being developed,
again the need for a higher-level abstraction emerged. In response, Qiskit introduced
the primitives interface, which is optimized for two core tasks in quantum algorithm development:
expectation value estimation (`Estimator`) and circuit sampling (`Sampler`). The goal is once
again to help developers to focus more on innovation and less on data conversion. The primitives interface supersedes the `backend.run` interface, since `Sampler` provides the same direct hardware access that was offered by `backend.run`.

## What is a primitive?

Computing systems are built on multiple layers of abstraction. Abstractions allow you to focus on a
particular level of detail relevant to the task at hand. The closer you get to the hardware,
the lower the level of abstraction you need (for example, you might need to move or manipulate data at the CPU instruction level). The more complex the task you want to perform,
the higher-level the abstractions will be (for example, you could be using a programming library to perform
algebraic calculations).

In this context, a *primitive* is the smallest processing instruction, the simplest building block from which
one can create something useful for a given abstraction level.

The recent progress in quantum computing has increased the need to work at higher levels of abstraction.
As the field moves toward larger quantum processing units (QPUs) and more complex workflows, the focus shifts from interacting with individual
qubit signals to viewing quantum devices as systems that perform necessary tasks.

The two most common tasks for quantum computers are sampling quantum states and calculating expectation values.
These tasks motivated the design of the Qiskit primitives: **Estimator** and **Sampler**.

- Estimator computes expectation values of observables with respect to states prepared by quantum circuits.
- Sampler samples the output register from quantum circuit execution.

In short, the computational model introduced by the Qiskit primitives moves quantum programming one step closer
to where classical programming is today, where the focus is less on the hardware details and more on the results
you are trying to achieve.

## Primitive definition and implementations

There are two types of Qiskit primitives: the base classes, and their implementations. The Estimator and Sampler primitives are defined by open-source primitive base classes that live in the Qiskit SDK (in the  [`qiskit.primitives`](/docs/api/qiskit/primitives) module). Providers (such as IBM Quantum Compute Service) can use these base classes to derive their own Sampler and Estimator implementations.  Most users will interact with provider implementations, not the base primitives.

### Base classes

The `Base` primitives are abstract classes that define a common interface for implementing primitives. All other classes in the [`qiskit.primitives`](/docs/api/qiskit/primitives) module inherit from these base classes. Developers should use these if they are interested in creating their own primitives-based execution model for a specific provider. These classes might also be useful for those who want to do highly customized processing and find that the existing primitives implementations are too simple for their needs. General users will not directly use the base classes.

[`BaseEstimatorV1`](/docs/api/qiskit/qiskit.primitives.BaseEstimatorV1) and [`BaseSamplerV1`](/docs/api/qiskit/qiskit.primitives.BaseSamplerV1) - Although the V1 primitives are still usable, these guides focus on the V2 primitives because they are the latest and are more commonly used.

[`BaseEstimatorV2`](/docs/api/qiskit/qiskit.primitives.BaseEstimatorV2) and [`BaseSamplerV2`](/docs/api/qiskit/qiskit.primitives.BaseSamplerV2) - The Qiskit reference primitives follow these interface specifications.

<span id="implementations"></span>
### Implementations

All primitives are created from the base classes; therefore, they have the same general structure and usage.  For example, the format of the input for all Estimator primitives is the same.  However, there are differences in implementations that make them unique.

These are implementations of the primitives base classes:

- The [IBM Quantum primitives](/docs/guides/qiskit-runtime-primitives), [`EstimatorV2`](/docs/api/qiskit-ibm-runtime/estimator-v2) and [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2), provide a more sophisticated implementation (for example, by including error mitigation) as a cloud-based service. This implementation of the base primitives is used to access IBM Quantum&reg; hardware.

- [`StatevectorEstimator`](/docs/api/qiskit/qiskit.primitives.StatevectorEstimator) and [`StatevectorSampler`](/docs/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Reference implementations of the primitives that use the simulator built into Qiskit. They are built with the Qiskit [`quantum_info`](/docs/api/qiskit/quantum_info#quantum-information) module, producing results based on ideal statevector simulations.  They are accessed through Qiskit. See [Exact simulation with Qiskit SDK primitives](/docs/guides/simulate-with-qiskit-sdk-primitives) for usage details.

- [`BackendEstimatorV2`](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) and [`BackendSamplerV2`](/docs/api/qiskit/qiskit.primitives.BackendSamplerV2) - You can use these classes to “wrap” any quantum computing resource into a primitive. This lets you write primitive-style code for providers that don’t yet have a primitives-based interface. These classes can be used just like the regular Sampler and Estimator, except they should be initialized with an additional `backend` argument for selecting which quantum computer to run on. They are accessed by using Qiskit. See the [backend primitives](/docs/guides/get-started-with-backend-primitives) guide for more information.

## Options

You can pass options to primitives to customize them to meet your needs. While the interface of the primitives' `run()`  method is common across all implementations, their options are not. Consult the API references for a specific primitive implementation to learn about the options it supports.

For example, refer to the [Estimator options](/docs/guides/estimator-options) and [Sampler options](/docs/guides/sampler-options) topics to learn about options for the IBM Quantum primitives, or see the [Qiskit Aer API references](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) for the Qiskit Aer primitives options.

## Benefits of Qiskit primitives

With primitives, Qiskit users can write quantum code for a specific QPU without having to explicitly
manage every detail. Also, because of the additional layer of abstraction, you might be able to more easily
access advanced hardware capabilities of a given provider. For example, with IBM Quantum primitives,
you can take advantage of the latest advancements in error mitigation and suppression by toggling options such as the primitive's [`resilience_level`](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level), rather than building your own implementation of these techniques.

For hardware providers, implementing primitives natively means you can provide your users with a more “out-of-the-box”
way to access your hardware features such as advanced post-processing techniques. It is therefore easier for your users to benefit from your hardware's best capabilities.

## Next steps

> **Tip:** - Understand the [primitive input and output](/docs/guides/primitive-input-output).
>     - Review detailed [examples](/docs/guides/simulate-with-qiskit-sdk-primitives).
>     - Practice with primitives by working through the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
>     - Review [Create a provider](/docs/guides/create-a-provider) to learn how to implement your own Sampler and Estimator primitives.
>     - See the [API references](/docs/api/qiskit/primitives).
>     - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
>     - Learn about the [IBM Quantum primitives](/docs/guides/qiskit-runtime-primitives), which are used for running circuits on IBM QPUs.